# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vedika1304-05/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import pandas as pd
import json
!git clone https://github.com/Vedika1304-05/flyrank-internship-ml.git
%cd flyrank-internship-ml

Cloning into 'flyrank-internship-ml'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 180 (delta 81), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (180/180), 1.93 MiB | 12.52 MiB/s, done.
Resolving deltas: 100% (81/81), done.
/content/flyrank-internship-ml


In [4]:
!python scripts/run_all.py



▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-internship-ml/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-internship-ml/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-internship-ml/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-internship-ml/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/flyrank-internship-ml/outputs/refresh_queue.csv
Wrote model report: /content/flyr

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

The ML model which we are trying to build here is a scoring based model whose output produces a ranking, which is used to identify the top 50 pages that the content team must refresh first.
We aren't doing any type of clutering here. We're just assigning a score (number) to each of the content items (pages) in our dataset and then ranking all these pages using the score given to them. We are using the features & observations that we have for a particular page & trying to assign that page a score based on these.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Prediction : Whether a page's traffic will decline in the next 30 days or so, instead of just finding out whether it's currently trending down.

Label comes from observed future outcome (real traffic measured after the fact, from the daily table) & not a rule applied to the current moment, which is what the starter's weaker proxy does.
We can use any feature from the available list of features to calculate the measured-outcome label, instead of just using the already available features (for eg. trend_direction) to form labels based on a pre-defined rule.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[df["impressions_prev_30d"] > 0].copy()

DECLINE_THRESHOLD = -0.20
df["measured_pct_change"] = (
    (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"]
)
df["measured_decline_label"] = df["measured_pct_change"] <= DECLINE_THRESHOLD

agreement = (df["measured_decline_label"] == (df["trend_direction"] == "down")).mean()
print(f"Agreement between trend_direction rule and prev/last 30d comparison: {agreement*100:.1f}%")
print("-> Near-perfect agreement suggests trend_direction is COMPUTED from these")
print("   same two already-recorded windows — not an independent future observation.")
print("   Both windows already exist in this single snapshot, so this is still a")
print("   same-file comparison, not a genuine prior-window -> later-window forecast")
print("   built around a real decision-point cutoff.")

Agreement between trend_direction rule and prev/last 30d comparison: 99.8%
-> Near-perfect agreement suggests trend_direction is COMPUTED from these
   same two already-recorded windows — not an independent future observation.
   Both windows already exist in this single snapshot, so this is still a
   same-file comparison, not a genuine prior-window -> later-window forecast
   built around a real decision-point cutoff.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The one success metric that we will use for evaluation is 'Precision@K', specifically Precision@50. It gives the % actually declining pages of the top 50 pages flagged as declining.
Another important metric worth considering is % of at-risk traffic caught in the top-K, i.e. out of all the declining pages, how many are caught in the top-K? This is useful because a model can be highly precise in terms of finding out the declining pages in top-50. However, it may not differentiate between a high-traffic & low-traffic page, as it is not trained for it. So, even though a page has low-traffic but is declining, it gets a higher rank in the top 50 pages, than another more important page with high traffic getting declined.

"Good" here is a relative term and not an absolute threshold. If we are using the measured (observed)-outcome label for our analysis then there are 3 possible options for calculating precision@50:
1. Simple base rate : Just counting the no. of declining pages out of the total pages to get a %.
2. Random ranking, averaged over many shuffles : Randomly rank the pages, find their precision@50 and then take an avg. for 1000 such trials.
3. Ranking based on traffic size : pages with higher traffic are ranked first.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[df["impressions_prev_30d"] > 0].copy()

DECLINE_THRESHOLD = -0.20
df["measured_pct_change"] = (
    (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"]
)
df["measured_decline_label"] = (df["measured_pct_change"] <= DECLINE_THRESHOLD).astype(int)

CAPACITY = 50

# STEP 1: the base rate — just count, no ranking at all
base_rate = df["measured_decline_label"].mean()

# STEP 2: prove random ranking converges to that base rate, over many trials
n_trials = 1000
random_precisions = [
    df.sample(frac=1.0, random_state=i).head(CAPACITY)["measured_decline_label"].mean()
    for i in range(n_trials)
]
random_precisions = np.array(random_precisions)

# STEP 3: sort deterministically by prior traffic size
p_traffic = df.sort_values("impressions_prev_30d", ascending=False) \
              .head(CAPACITY)["measured_decline_label"].mean()

print(f"Base rate (no ranking):                 {base_rate:.3f}")
print(f"Random ranking (avg of {n_trials} trials):    {random_precisions.mean():.3f}")
print(f"Sorted by prior traffic size:            {p_traffic:.3f}")

Base rate (no ranking):                 0.613
Random ranking (avg of 1000 trials):    0.613
Sorted by prior traffic size:            0.500


This shows that even the base rate  (obtained through random ranking)is still better than the precision value obtained through traffic-size based ranking. So, we should be using this baseline value of 61.3% as a threshold for our model. The actual precision value obtained from our model should be >61.3% atleast.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content item (content_id), belonging to one client (client_id), with that page's own metrics aggregated over a fixed 90-day window. Every client appears multiple times in the dataset — once per unique content item it owns — so client_id repeats, but content_id never does.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Does the row count match the count of unique content_ids?
print(f"Total rows: {len(df):,}")
print(f"Unique content_id values: {df['content_id'].nunique():,}")
print(f"Are rows == unique content_ids? {len(df) == df['content_id'].nunique()}")

# One real row, to see what a single unit actually contains
print(df.iloc[0][["content_id", "client_id", "impressions_90d", "trend_direction"]])

# Rule out "one row per client"
pages_per_client = df.groupby("client_id")["content_id"].nunique()
print(f"Pages per client — min: {pages_per_client.min()}, median: {pages_per_client.median():.0f}, "
      f"max: {pages_per_client.max()}")

# Rule out "one row per day" — check for a genuine daily dimension
real_date_cols = [c for c in df.columns if "report_date" in c.lower() or "timestamp" in c.lower()]
print(f"Per-day date columns present: {real_date_cols if real_date_cols else 'NONE'}")

Total rows: 30,000
Unique content_id values: 30,000
Are rows == unique content_ids? True
content_id         content_304f48230142
client_id             client_f369cb89fc
impressions_90d                    3803
trend_direction                    down
Name: 0, dtype: object
Pages per client — min: 3, median: 567, max: 7008
Per-day date columns present: NONE


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single variable tops out at 0.600. Manually combining two well-chosen variables nudges this to 0.620 — a small gain. But naively combining all variables with equal weight actually performs worse (0.360) than doing nothing more than sorting by position alone, proving that "more signals" only helps if they're weighted correctly. The random forest's 0.740 comes from automatically learning both the right weights and nonlinear interactions between variables — something no fixed, hand-written weighting scheme achieved in this test, even trying every 2-variable pair and the full 6-variable average.

Hence, a sinple if-statement won't work for defining the labels as whether or not a page will decline depends on a combination of multiple features (with right weights) than a single feature. A single if condition can only draw one straight cutoff line through one dimension — but the real boundary between "will decline" and "won't" isn't a straight line in any single dimension, it's a curved, combined boundary across several.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json
import pandas as pd
from itertools import combinations

# Load the model predictions (has the train/test split + model probabilities)
# and the baseline queue (has all the raw feature columns)
preds = pd.read_csv("data/processed/model_predictions.csv")
base  = pd.read_csv("data/processed/baseline_refresh_queue.csv")

# Restrict to the held-out TEST split only — same rows the model itself was scored on,
# so every comparison below is fair (apples-to-apples)
test_ids = set(preds.loc[preds["split"] == "test", "content_id"])
test = base[base["content_id"].isin(test_ids)].merge(
    preds[["content_id", "best_model_probability"]], on="content_id"
).copy()

CAPACITY = 50

def precision_at_k(data, score_col, k=CAPACITY, ascending=False):
    """Sort by score_col, take the top k, and report what fraction are truly declining."""
    return data.sort_values(score_col, ascending=ascending).head(k)["is_declining_label"].mean()

# Candidate signals to test (excluding trend_pct - confirmed leakage, since
# trend_direction is literally thresholded from it).
# direction: True = a LOWER value is more "concerning" (sort ascending),
#            False = a HIGHER value is more "concerning" (sort descending)
signals = {
    "avg_position":          False,
    "engagement_rate":        True,
    "content_age_days":       False,
    "word_count":             True,
    "ctr":                    True,
    "days_since_last_update": False,
}

# Rank-normalize each signal to a 0-1 scale (percentile rank) so they're
# comparable and combinable, regardless of their original units
ranked = test.copy()
for col, ascending in signals.items():
    ranked[f"{col}_rank"] = ranked[col].rank(ascending=not ascending, pct=True)

# --- STEP 1: Best single variable ---
single_results = {c: precision_at_k(test, c, ascending=asc) for c, asc in signals.items()}
best_single_name = max(single_results, key=single_results.get)
print("Single-variable results:")
for name, p in sorted(single_results.items(), key=lambda x: -x[1]):
    print(f"  {name:28s} Precision@50: {p:.3f}")
print(f"\nBest single: {best_single_name} -> {single_results[best_single_name]:.3f}\n")

# --- STEP 2: Every 2-variable combo (simple average of rank scores) ---
pair_results = {}
for c1, c2 in combinations(signals.keys(), 2):
    ranked["combo_score"] = (ranked[f"{c1}_rank"] + ranked[f"{c2}_rank"]) / 2
    pair_results[f"{c1} + {c2}"] = precision_at_k(ranked, "combo_score")

best_pair_name = max(pair_results, key=pair_results.get)
print("Top 5 two-variable combos:")
for name, p in sorted(pair_results.items(), key=lambda x: -x[1])[:5]:
    print(f"  {name:45s} Precision@50: {p:.3f}")
print(f"\nBest pair: {best_pair_name} -> {pair_results[best_pair_name]:.3f}\n")

# --- STEP 3: All 6 variables, naively averaged ---
ranked["all_combo_score"] = ranked[[f"{c}_rank" for c in signals]].mean(axis=1)
p_all = precision_at_k(ranked, "all_combo_score")
print(f"All 6 variables (equal-weighted average): Precision@50: {p_all:.3f}\n")

# --- STEP 4: Random forest model ---
res = json.load(open("outputs/model_results.json"))
rf_precision = res["models"]["random_forest"]["precision_at_50"]
# --- STEP 5: Full progression summary ---
print(f"{'Method':50s}{'Precision@50':>15}")
print("-" * 65)
print(f"{'Best single variable (' + best_single_name + ')':50s}{single_results[best_single_name]:>15.3f}")
print(f"{'Best 2-variable combo':50s}{pair_results[best_pair_name]:>15.3f}")
print(f"{'All 6 variables (naive equal-weight average)':50s}{p_all:>15.3f}")
print(f"{'Random forest (learns weights automatically)':50s}{rf_precision:>15.3f}")

Single-variable results:
  avg_position                 Precision@50: 0.600
  engagement_rate              Precision@50: 0.500
  content_age_days             Precision@50: 0.460
  word_count                   Precision@50: 0.460
  ctr                          Precision@50: 0.300
  days_since_last_update       Precision@50: 0.200

Best single: avg_position -> 0.600

Top 5 two-variable combos:
  avg_position + engagement_rate                Precision@50: 0.620
  avg_position + ctr                            Precision@50: 0.600
  avg_position + days_since_last_update         Precision@50: 0.560
  content_age_days + ctr                        Precision@50: 0.500
  engagement_rate + content_age_days            Precision@50: 0.480

Best pair: avg_position + engagement_rate -> 0.620

All 6 variables (equal-weighted average): Precision@50: 0.360

Method                                               Precision@50
-----------------------------------------------------------------
Best single varia

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.